In [ ]:
import pandas as pd

# ???? labeledTrainData.tsv ???????? ??????, ??????? ?????????? ?? ???????
train = pd.read_csv('labeledTrainData.tsv', header=0, delimiter="\t", quoting=3)

# ???? testData.tsv ???????? ??????, ?? ??????? ????? ?????? ????????????
test = pd.read_csv('testData.tsv', header=0,  delimiter="\t", quoting=3)

# ??????? ???????? ?? unlabeledTrainData.tsv
# ??? ??????? ????? ??? ?????, ? ?????? ?????? ????????? ??? ??????
# ??? ?? ???????????? ?? ???? ???????????? ??? ?? ??????? ??? ??????? ?? ??????? ?????
# ? ??? ????????????? ?????? ???? ?? ??????, ??? ? ?????????
# ? ?????? ??????? ??? ? ???????? word2vec ?? ??????? ?????? ?????? ?????? ? "????"
unsup = pd.read_csv('unlabeledTrainData.tsv', header=0,  delimiter="\t", quoting=3)

??????? ??????? ??????? ??? ??????????????? ??????:
* ?????? n't ?? not (?? ???????)
* ???????? ? ??????? ????????
* ????? ?? ?????
* ??????? ????-????? (?? ???????)

In [ ]:
from bs4 import BeautifulSoup
import re
#from nltk.corpus import stopwords

def text_to_wordlist(text):
    text = re.sub('n\'t', ' not', text)
    
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.lower().split()

    #stops = set(stopwords.words("english"))

    return words

In [ ]:
# ???????????? ?????? ???????

text_to_wordlist("""
Tyrone Garland <s>(born 1992)</s> is an American professional basketball player 
who last played with the National <b>Basketball</b> League of Canada's Mississauga Power
""")

# ????????? ??? ??????????? ????? list, ???????? ???????? - ?????

??????? ??????? ??????? ??? ??????????????? ??????:
* ??????? html ????
* ?????????? ??????? ?? ???????????
* ?????? ??????????? ????? ?? ????? (???????? ???? ?????????? ???????)

In [ ]:
import nltk.data

def text_to_sentences(text):
    text = BeautifulSoup(text).get_text()
    
    tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
    raw_sentences = tokenizer.tokenize(text.strip())
    
    sentences = []
    for raw_sentence in raw_sentences:
        if len(raw_sentence) > 0:
            sentences.append(text_to_wordlist(raw_sentence))

    return sentences

In [ ]:
# ???????????? ?????? ???????

text_to_sentences("""
Sherlock Holmes is a four-act play written by <p/> William Gillette and Sir Arthur Conan Doyle, 
based on Conan Doyle's eponymous character. It drew material from the stories 
<s>"A Scandal in Bohemia"</s>, "The Final Problem", and A Study in Scarlet, pitting Holmes 
against Professor Moriarty and reinventing the character of Irene Adler as a new love 
interest named Alice Faulkner. This play introduced the phrase "Elementary, my dear Watson" 
and Holmes' curved pipe.
""")


# ????????? ??? ??????????? ????? list, ???????? ???????? - list, ???????? ???????? - ?????
# ?.?. ???????? list list'?? (?????? ???????)

??? ??????? ??????? ??? ????????? ?????? ? ??????? ?????????

In [ ]:
%%time
sentences = []

for review in train["review"]:
    sentences += text_to_sentences(review)

In [ ]:
%%time
for review in unsup["review"]:
    sentences += text_to_sentences(review)

In [ ]:
%%time
for review in test["review"]:
    sentences += text_to_sentences(review)

In [ ]:
# ? ????? ????? sentences ????? list list'?? (?????? ???????) - ??? ? ?????? ????.
# (?????????) ?????? ??????? ?????? sentences - ???????????, ?? ?????????????? ? ???? ?????? ???? - ?????? ??????

# ??????? ?????????? ????????? ????? ??????? (??? ?? - ?????????? ??????????? ?? ???? ??????)
print len(sentences)

In [ ]:
# ? ??? ?? ????????? ?? ??? ??????
print '\t ?????? ??????? ???????'
print sentences[0]
print 
print '\t ?????? ??????? ???????'
print sentences[1]

?????? ?? ?????? ?????? Word2Vec

In [ ]:
%%time
# ??????????? ??????????????? ??????? ?? ?????? gensim, ??????? ?????? ???? ??????????

from gensim.models.word2vec import Word2Vec 

# ?????? ??????????, ??????? ????? ?????? ?? ?????? ???????
num_features = 300  # ???????? ??????????? ??????? ??????? ?????
min_word_count = 5  # ??????????? ??????????? ?????, ????? ??? ?????? ? ??????
num_workers = 8     # ?????????? ???? ?????? ??????????, ???? ????????? ???????? ? ????????? ???????
context = 10        # ?????? ???? 
downsampling = 1e-3 # ?????????? ??????? ??????

model = Word2Vec(sentences, workers=num_workers, size=num_features,
                 min_count=min_word_count, window=context, sample=downsampling)

In [ ]:
# ???????????? ???? ??????. ?? ?????? ????? ??????? ??????, ?? ??? ?????? ???????? ??????? ?????? ?????
model.init_sims(replace=True)

???????????? ??????, ???????? ????????????? ??????? ????? ? ????????????? ???????????? (????? ???????? "??????" ????????????? ????????????)

????????? ?????????? ??????: QUEEN + MAN - KING = ? 

In [ ]:
model.most_similar(positive=['queen', 'man'], negative=['king'])

?????????? WOMAN, ??? ??????? :) 

In [ ]:
# ????????? ?????, ??????? ????? ?????? ?? ????? MOVIE

model.most_similar('movie')

# ????????? ? ?????? ?????

In [ ]:
model.most_similar('awful')

In [ ]:
model.most_similar('big')

In [ ]:
model.most_similar('mail')

In [ ]:
# ??? ?? ???????? ?? ??????? doesnt_match, ??? ??????? ?????? ????? ? ??????

model.doesnt_match(['berlin', 'moscow', 'africa', 'riga', 'madrid'])

In [ ]:
model.doesnt_match(['man', 'woman', 'child', 'dog'])

In [ ]:
# ??????? ?????????? ?????? ?????? ?? ????
# ??? ?????
print len(model['moon'])

# ? ??? ??????
print model['moon']

In [ ]:
# ??????? - ??? ????? ??????? ????????? ? ?????? ????? ??????????? ???
model.index2word

?.?. ?? ????? ???????????????? ?? ?????, ? ??????, ???? ????????? ?????? ? ??????? (??????????? ? ???? ???)
???? ?? ??????? ??????? - ??????? ??? ??????? ???? ???????? ? ????? ? ???????? ?? ????? ???????? ????.
??????? ???????, ???????:
* ??????? ??????? ?????? - ??? ????? ?????????????? ??????
* ???? ?? ???? ?????? ? ??????, ???? ????? ???? ? ??????:
  * ??????????? ??????? ????
  * ???????? ?????? ????? ? ??????????????? ???????
* ??????? ??? ?????????? ?? ????? ???? ?? ???????, ??????? ??????? ?? ?????????? ? ??????????????? ???????

In [ ]:
import numpy as np

def text_to_vec(words, model, size):
    text_vec = np.zeros((size,), dtype="float32")
    n_words = 0

    index2word_set = set(model.index2word)
    for word in words:
        if word in index2word_set:
            n_words = n_words + 1
            text_vec = np.add(text_vec, model[word])
    
    if n_words != 0:
        text_vec /= n_words
    return text_vec

??????? ???????, ??????? ?? ????? ???????? ?????? ???? ???????, ? ?? ?????? ?????? ?????? ?????? ??????? ?????? - ??? ???????? ????????????? ????????

In [ ]:
def texts_to_vecs(texts, model, size):
    texts_vecs = np.zeros((len(texts), size), dtype="float32")
    
    for i, text in enumerate(texts):
        texts_vecs[i] = text_to_vec(text, model, size)

    return texts_vecs

??????? ??? ??????? texts_to_vecs ????????? ?? ?????? ??????, ? ?????? ???? ???? ??????. 

(!!!) ????????: ?? ?????? ??????? (??? ??? ??????? ?????? ?? ???????????, ? ??????????? ?? ?????), ? ??????? ???????? ??????

?? ? ??? ???? ???????, ??????? ????????? 1) ????? ? ?????? ???????????, 2) ??????????? ? ?????? ????

????? ??????????, ??? ????? ???????????? 2?? ???????, ?? ???????? ????? ?? ??????????, ?????? ??? ???? ? ??? ????????? ???? ? ?????? ???????

In [ ]:
# ???????? ?????, ? python ???? ??????????? ?????????? ?????????? ?????? ? ??????????, ??? ??????

temp_list = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
print sum(temp_list, [])

# ????? :)

In [ ]:
# ?? ?? ????? ???????????? ?????????????, ???? ??? ??????? text_to_sentences ?????????? ?????? ???????

list_of_list_of_words = text_to_sentences("""
Sherlock Holmes is a four-act play written by <p/> William Gillette and Sir Arthur Conan Doyle, 
based on Conan Doyle's eponymous character. It drew material from the stories 
<s>"A Scandal in Bohemia"</s>, "The Final Problem", and A Study in Scarlet, pitting Holmes 
against Professor Moriarty and reinventing the character of Irene Adler as a new love 
interest named Alice Faulkner. This play introduced the phrase "Elementary, my dear Watson" 
and Holmes' curved pipe.
""")

print '\t ????? list_of_list_of_words'
print len(list_of_list_of_words)
print '\t ?????? ??????? list_of_list_of_words'
print list_of_list_of_words[0]

list_of_words = sum(list_of_list_of_words, [])

print '\t ????? list_of_words'
print len(list_of_words)
print '\t ?????? ??????? list_of_words'
print list_of_words[0]

In [ ]:
# ????????????? ????????, ??????? ??? ???? ??????? ?? train
train_like_word_list = [sum(text_to_sentences(text), []) for text in train['review']]

In [ ]:
train_vecs = texts_to_vecs(train_like_word_list, model, num_features)

In [ ]:
# ??????? ???? ????? ??? test
test_like_word_list = [sum(text_to_sentences(text), []) for text in test['review']]

In [ ]:
test_vecs = texts_to_vecs(test_like_word_list, model, num_features)

In [ ]:
# ????????????? train_vecs, test_vecs, train["sentiment"] 
#    ??? ??????? ????? ????????? ???????, ??????? ????? ???????? ???????, ?????? ??? ????????? ??????? ??????????????

# ??????????? ????????? ??? ? ????? ??????
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, n_jobs=8)
forest = forest.fit(train_vecs, train["sentiment"])
predict = forest.predict(test_vecs)

# ? ??? ?????? ??????

?? ??? ???? ??? ????????? ??????????????? ??????? ?? ??????????? ???????, ? ????? ?????? ????????.

???????????? ??? ????? ?? 1000 ???????.

In [ ]:
# ? model.syn0 ????????? ??? ???????. ?????????????? ??!

print '?????? ', model.syn0.shape
print '?????? ???????? ', model.syn0

In [ ]:
%%time
# ?????????????? ??? ?????. 

from sklearn.cluster import KMeans

word_vectors = model.syn0
# ????? ????????? ????????? ? 10000. ??? ????? ????? ??? "?????????? ????". ??? ??????? ?????? ????? ???????? ??????
num_clusters = 10000

# ?????? ?????????????, ???????? ??? ??????? ?????, ?????????? ???????? (?? ???? ????) ?????,
#   ??? ??? ????? ?????????? ??????????????? ?????. ????? ??????? ?? ????.
kmeans_clustering = KMeans(n_clusters=num_clusters)
idx = kmeans_clustering.fit_predict(word_vectors)

# ? idx ????? ????????? ?????? ??????? ??? ??????? ?????

In [ ]:
# ???????? ????????? dict (???????): ????? -> ?????
word_centroid_map = dict(zip(model.index2word, idx))

In [ ]:
# ??????? ?????????????? ?????? 10 ??????? ? ????????? ?? ???????????? ???????????? ?????????????

for cluster in xrange(0,10):
    print cluster
    words = []
    for i in xrange(0, len(word_centroid_map.values())):
        if word_centroid_map.values()[i] == cluster:
            words.append(word_centroid_map.keys()[i])
    print words

?????????? ????????? ???????? ? ?????? ? ?????????? ????. ?????????? 2 ???????
* 1) ????????? ??? ??????? ?????? ?????? ??????? ??? ???? ??????????? ? ?????? ????????. ?.?. ??? ??????? ?????? ????? ?????? ??????? ??????? ????? ?????????, ?????????? ??????? ????? ??????????? ?????
* 2) ????????? ??????????? ??????????? ???????? ??????? ?? ??????? ????????? ???? ?????????.

In [ ]:
# ??? ??????? ?????? ??????? ???????, ?? ???? ??????? ????????? ?????, ?????????????? ? ???? ?????? ???? ??? ????
#    ??????? ? ????? ???????? ????????? ?????? ????? 
#    ? ??????????? ??????????????? ?????? (????????????? ?? ???? ???????) ?? 1


def create_bag_of_centroids(wordlist, word_centroid_map, num_centroids):
    bag_of_centroids = np.zeros(num_centroids, dtype="float32")
    set_word_centroid_map = set(word_centroid_map.keys())
    
    for word in wordlist:
        if word in set_word_centroid_map:
            index = word_centroid_map[word]
            bag_of_centroids[index] += 1
    
    return bag_of_centroids

In [ ]:
# ?? ??? ????? ??? ?? ??? ?????? ??????, ? ??? ???? ??????? ????????? ? ???????? ???????
# ??????? ???

train_vecs_centroids = np.zeros((train["review"].size, num_clusters), dtype="float32")

for i, text in enumerate(train_like_word_list):
    train_vecs_centroids[i] = create_bag_of_centroids(text, word_centroid_map, num_clusters)

test_vecs_centroids = np.zeros((test["review"].size, num_clusters), dtype="float32")

for i, text in enumerate(test_like_word_list):
    test_vecs_centroids[i] = create_bag_of_centroids(text, word_centroid_map, num_clusters)
    
# ??????????? ????? ??????? ??? ????????? ? ???????? ???????, ??? ????? ???????? ??? ????????? ? ???????? ??????? ?????.

In [ ]:
# ??????????? ????????? ??? ?? ?????????? ????????

forest = RandomForestClassifier(n_estimators=100, n_jobs=8)
forest = forest.fit(train_vecs_centroids, train["sentiment"])
predict = forest.predict(test_vecs_centroids)

---

In [ ]:
# ? ?????? ????????? ????????? ????????? ?? ??????????

train_vecs_centroids_dist = np.zeros((train["review"].size, num_clusters), dtype="float32")

from scipy.spatial import distance

for i, vec in enumerate(train_vecs):
    for j, center in enumerate(kmeans_clustering.cluster_centers_):
        train_vecs_centroids_dist[i][j] = distance.euclidean(vec, center)

In [ ]:
train_vecs_centroids_dist.shape

In [ ]:
test_vecs_centroids_dist = np.zeros((test["review"].size, num_clusters), dtype="float32")

for i, vec in enumerate(test_vecs):
    for j, center in enumerate(kmeans_clustering.cluster_centers_):
        test_vecs_centroids_dist[i][j] = distance.euclidean(vec, center)

In [ ]:
test_vecs_centroids_dist.shape

In [ ]:
# ??????????? ????????? ??? ?? ?????????? ???????? (??? ?? ??????????)

forest = RandomForestClassifier(n_estimators=100, n_jobs=8)
forest = forest.fit(train_vecs_centroids_dist, train["sentiment"])
predict = forest.predict(test_vecs_centroids_dist)